In [ ]:
import sys
from pathlib import Path

import torch
from torchvision import models

repo_root = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(repo_root))

from tutorial._infra import torch_frontend as torch_nb

torch_nb.print_setup()

class MobileNetWrapper(torch.nn.Module):
    def __init__(self):
        super().__init__()
        try:
            self.m = models.mobilenet_v3_small(weights=None)
        except TypeError:
            self.m = models.mobilenet_v3_small(pretrained=False)
        self.m.eval()

    def forward(self, x):
        return self.m(x)

model = MobileNetWrapper()


In [ ]:
import torch
from torch_mlir import fx

mobilenet_module = MobileNetWrapper().eval()
example_input = torch.randn(1, 3, 224, 224, dtype=torch.float32)

torch_module = fx.export_and_import(mobilenet_module, example_input, func_name="kernel")
torch_ir = torch_module.operation.get_asm()

torch_file = torch_nb.ARTIFACTS_DIR / "mobilenet_v3_small_torch.mlir"
torch_file.write_text(torch_ir)

print(f"Wrote Torch dialect IR → {torch_file.resolve()}")
print(torch_ir[:100])

In [ ]:
from pathlib import Path

pipeline = (
    "builtin.module("
    "torch-function-to-torch-backend-pipeline,"
    "torch-backend-to-linalg-on-tensors-backend-pipeline,"
    "torch-verify-linalg-on-tensors-backend-contract"
    ")"
)

torch_ir = torch_nb.ARTIFACTS_DIR / "mobilenet_v3_small_torch.mlir"
linalg_ir = torch_nb.ARTIFACTS_DIR / "mobilenet_v3_small_linalg.mlir"

torch_nb.run(
    [
        torch_nb.torch_mlir_opt,
        torch_ir,
        f"-pass-pipeline={pipeline}",
        "-o",
        linalg_ir,
    ]
)

print(f"Wrote Linalg IR → {linalg_ir}")
linalg_txt = linalg_ir.read_text()
print(linalg_txt[:800])

In [ ]:
from tutorial._infra import cinm_frontend as cinm_nb

cinm_pipeline = (
    "builtin.module("
    "func.func(cinm-cleanup-linalg,im2col-to-matmul)"
    ")"
)

mobilenet_linalg_im2col_clean = torch_nb.ARTIFACTS_DIR / "mobilenet_v3_small_linalg_im2col_clean.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        linalg_ir, 
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        mobilenet_linalg_im2col_clean,
    ]
)

print(mobilenet_linalg_im2col_clean.read_text()[:2000])

In [ ]:
cinm_pipeline = (
    "builtin.module("
    "func.func(convert-linalg-to-cinm)"
    ")"
)

mobilenet_cinm0 = torch_nb.ARTIFACTS_DIR / "mobilenet_v3_small_cinm0.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        mobilenet_linalg_im2col_clean,  # from previous step
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        mobilenet_cinm0,
    ]
)

print(mobilenet_cinm0.read_text()[:2000])

In [ ]:
cinm_pipeline = (
    "builtin.module("
    "cinm-annotate-tiles{ops=gemv tile-sizes=32x32},"
    "cinm-annotate-tiles{ops=gemv tile-sizes=32x32},"
    "cinm-annotate-tiles{ops=gemm tile-sizes=32x32x32},"
    "cinm-annotate-tiles{ops=batch_gemv tile-sizes=32x32x32},"
    "cinm-annotate-tiles{ops=batch_gemm tile-sizes=32x32x32x32},"
    "cinm-annotate-tiles{ops=activate tile-sizes=32}"
    ")"
)

mobilenet_cinm1 = torch_nb.ARTIFACTS_DIR / "mobilenet_v3_small_cinm1.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        mobilenet_cinm0,  # from previous step
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        mobilenet_cinm1,
    ]
)

print(mobilenet_cinm1.read_text()[:2000])


In [ ]:
cinm_pipeline = (
    "builtin.module("
    "cinm-gemm-to-gemv{split-dim=2}"
    ")"
)

mobilenet_cinm2 = torch_nb.ARTIFACTS_DIR / "mobilenet_v3_small_cinm2.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        mobilenet_cinm1,  # from previous step
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        mobilenet_cinm2,
    ]
)

print(mobilenet_cinm2.read_text()[:2000])


In [ ]:
cinm_pipeline = (
    "builtin.module("
    "cinm-tiling"
    ")"
)

mobilenet_cinm3 = torch_nb.ARTIFACTS_DIR / "mobilenet_v3_small_cinm3.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        mobilenet_cinm2,  # from previous step
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        mobilenet_cinm3,
    ]
)

print(mobilenet_cinm3.read_text()[:2000])


In [ ]:
cinm_pipeline = (
    "builtin.module("
    "func.func(cinm-decompose-accum)"
    ")"
)

mobilenet_cinm4 = torch_nb.ARTIFACTS_DIR / "mobilenet_v3_small_cinm4.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        mobilenet_cinm3,  # from previous step
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        mobilenet_cinm4,
    ]
)

print(mobilenet_cinm4.read_text()[:2000])


In [ ]:
cinm_pipeline = (
    "builtin.module("
    "func.func(cinm-insert-quantization{ops=gemm,gemv qtype=i8 scale=0.03125 zp=0 rounding=nearest narrow-range=false})"
    ")"
)

mobilenet_cinm5 = torch_nb.ARTIFACTS_DIR / "mobilenet_v3_small_cinm5.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        mobilenet_cinm4,  # from previous step
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        mobilenet_cinm5,
    ]
)

print(mobilenet_cinm5.read_text()[:2000])

In [ ]:
cinm_pipeline = (
    "builtin.module("
    "lower-affine,"
    "func.func(cinm-relower)"
    ")"
)

mobilenet_cinm6 = torch_nb.ARTIFACTS_DIR / "mobilenet_v3_small_cinm6.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        mobilenet_cinm5,
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        mobilenet_cinm6,
    ]
)

print(mobilenet_cinm6.read_text()[:2000])


In [ ]:
cinm_pipeline = (
    "builtin.module("
    "func.func(linalg-generalize-named-ops,canonicalize,scf-for-loop-canonicalization),"
    "one-shot-bufferize{bufferize-function-boundaries}"
    ")"
)

mobilenet_cinm7 = torch_nb.ARTIFACTS_DIR / "mobilenet_v3_small_cinm7.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        mobilenet_cinm6,
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        mobilenet_cinm7,
    ]
)

print(mobilenet_cinm7.read_text()[:2000])


In [ ]:
cinm_pipeline = (
    "builtin.module("
    "cinm-memory-cleanup,"
    "func.func(convert-cinm-to-cim,cim-mark-relower{ops=add,relu}),"
    "cinm-memory-cleanup"
    ")"
)

mobilenet_cinm8 = torch_nb.ARTIFACTS_DIR / "mobilenet_v3_small_cinm8.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        mobilenet_cinm7,
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        mobilenet_cinm8,
    ]
)

print(mobilenet_cinm8.read_text()[:2000])


In [ ]:
cinm_pipeline = (
    "builtin.module("
    "func.func(convert-cim-to-alpine,cim-cleanup-unsupported)"
    ")"
)

mobilenet_cinm9 = torch_nb.ARTIFACTS_DIR / "mobilenet_v3_small_cinm9.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        mobilenet_cinm8,
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        mobilenet_cinm9,
    ]
)

print(mobilenet_cinm9.read_text()[:2000])


In [ ]:
cinm_pipeline = (
    "builtin.module("
    "convert-alpine-to-func,"
    "func.func(convert-linalg-to-loops),"
    "func.func(affine-expand-index-ops),"
    "lower-affine,"
    "convert-scf-to-cf,"
    "expand-strided-metadata,"
    "func.func(affine-expand-index-ops),"
    "lower-affine,"
    "convert-vector-to-llvm,"
    "convert-math-to-llvm,"
    "convert-arith-to-llvm,"
    "convert-index-to-llvm,"
    "convert-to-llvm,"
    "func.func(llvm-request-c-wrappers),"
    "reconcile-unrealized-casts,"
    "canonicalize"
    ")"
)

mobilenet_cinm10 = torch_nb.ARTIFACTS_DIR / "mobilenet_v3_small_cinm10.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        mobilenet_cinm9,
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        mobilenet_cinm10,
    ]
)

print(mobilenet_cinm10.read_text()[:2000])


In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(repo_root))

from tutorial._infra import torch_frontend as torch_nb
from tutorial._infra import nbtools

mobilenet_cinm10 = torch_nb.ARTIFACTS_DIR / "mobilenet_v3_small_cinm10.mlir"
mobilenet_cinm10_ll = nbtools.mlir_translate(mobilenet_cinm10)
print(mobilenet_cinm10_ll.read_text()[:2000])